# M3L3 E00 — El problema del agente único
### Módulo 3 · Lecture 3 · Sistemas Multiagente

## ¿Qué vas a aprender hoy?
- ver por qué un agente único falla cuando mezcla dominios.
- Conectar el concepto con M3L2.
- Leer código pequeño con explicación previa.
- Interpretar resultados y checks.


## ¿Qué necesitás saber antes?

Venís de M3L2 con LangChain, LCEL, PromptTemplate, RAG con FAISS y memoria conversacional. En M3L3 usamos esas piezas para coordinar varios agentes.

> **Sistema multiagente:** arquitectura donde varias unidades especializadas colaboran bajo una política de coordinación.


## Instalación e imports

En un notebook productivo podrías instalar `langchain`, `langchain-openai` y `faiss-cpu`. Aquí usamos Python estándar para que el foco sea el diseño multiagente y no la API key.


In [ ]:
from typing import Callable, TypedDict, Literal
from dataclasses import dataclass, field
import json

print("Setup listo: usamos Python estándar para que el notebook pueda correr sin API key.")


## Sección 1 — ¿Qué es un dominio?

> **Dominio:** área de conocimiento con reglas, datos y vocabulario propios.

| Dominio | Preguntas típicas |
|---|---|
| HR | vacaciones, beneficios, licencias |
| Tech | VPN, contraseña, notebook |
| Billing | facturas, reembolsos, pagos |

Un agente único intenta cubrir todos esos temas desde un solo lugar.


Ahora cargamos una base de conocimiento mezclada. Esta celda no es todavía multiagente: solo prepara el escenario donde un único asistente ve todo junto.


In [ ]:
knowledge_base = {
    "hr": [
        "Vacaciones: cada empleado tiene 15 días hábiles por año.",
        "Beneficios: el seguro médico inicia el primer día de trabajo.",
        "Licencias: registrar el pedido en PeopleOps y avisar al manager.",
    ],
    "tech": [
        "VPN: reiniciar el cliente, validar MFA y abrir ticket si persiste.",
        "Contraseña: restablecer desde el portal de identidad.",
        "Notebook: reportar equipo dañado con número de serie.",
    ],
    "billing": [
        "Facturas: cargar comprobantes antes del día 25.",
        "Reembolsos: adjuntar recibo, monto y centro de costo.",
        "Pagos: se procesan los viernes por la tarde.",
    ],
}

KEYWORDS = {
    "hr": ["vacaciones", "beneficio", "seguro", "licencia"],
    "tech": ["vpn", "contraseña", "mfa", "notebook"],
    "billing": ["factura", "facturas", "reembolso", "pago", "recibo"],
}

def detect_domains(query: str) -> list[str]:
    text = query.lower()
    matches = [domain for domain, words in KEYWORDS.items() if any(word in text for word in words)]
    return matches or ["unknown"]

def retrieve(domain: str, query: str, k: int = 2) -> list[str]:
    docs = knowledge_base[domain]
    query_words = set(query.lower().replace("¿", "").replace("?", "").split())
    def score(doc: str) -> int:
        return sum(1 for word in query_words if word.strip(",.") in doc.lower())
    return sorted(docs, key=score, reverse=True)[:k]

mixed_docs = [doc for docs in knowledge_base.values() for doc in docs]
print("Documentos mezclados:", len(mixed_docs))


## Sección 2 — Construir el agente único

La función `single_agent(query)` recibe una consulta y devuelve un `dict` con dominio estimado, respuesta y tamaño de contexto.

El problema: la misma función clasifica, recupera contexto y responde.


In [ ]:
# TODO: completar el agente único.
# Debe recibir query: str y devolver query, predicted_domain, answer y context_size.
def single_agent(query: str) -> dict:
    return {"query": query, "predicted_domain": "TODO", "answer": "TODO", "context_size": 0}


## Sección 3 — Probar distintos dominios

Corremos seis consultas, dos por dominio, para mirar aciertos y errores. El resultado importante es la tabla, no una respuesta perfecta.


In [ ]:
consultas = [
    ("¿Cuántos días de vacaciones tengo?", "hr"),
    ("¿Cuándo empieza mi seguro médico?", "hr"),
    ("Mi VPN no conecta", "tech"),
    ("Olvidé mi contraseña", "tech"),
    ("¿Hasta cuándo cargo facturas?", "billing"),
    ("Necesito un reembolso", "billing"),
]
for query, expected in consultas:
    result = single_agent(query)
    print(f"{expected:8} | pred={result['predicted_domain']:8} | {query}")


## Checks automáticos

Los checks verifican el contrato mínimo del ejercicio. En Starter pueden fallar hasta completar los TODOs; en Resolution deben pasar.


In [ ]:
def run_checks():
    sample = single_agent("Mi VPN no conecta")
    assert isinstance(sample, dict)
    assert "answer" in sample and "predicted_domain" in sample
    print("Checks E00 OK")
run_checks()


## ¿Qué aprendiste hoy?

- Ver por qué un agente único falla cuando mezcla dominios.
- Separar responsabilidades vuelve el sistema más auditable.
- Los contratos explícitos hacen que el orquestador dependa menos de texto libre.

## Próximo ejercicio

Continuá con el siguiente notebook de M3L3 para agregar una pieza más de coordinación multiagente.
